# Filtering queries

In this exercise, you'll practice querying the 'datacamp-index' Pinecone index.
You'll connect to the index and query it using the vector provided
to retrieve similar vectors.
You'll also use metadata filtering to optimize your querying
and return the most relevant search results.

In [1]:
from DATA import vector
from pinecone_helper import get_index

index = get_index()

# Retrieve the MOST similar vector with the year 2024
query_result = index.query(
    vector=vector,
    filter={
        "year": {"$eq": 2024},
    },
    top_k=1,
    include_metadata=True,
)

print(query_result)

QueryResponse(matches=[ScoredVector(id='69', score=0.0277260877, values=[], metadata={'genre': 'horror', 'year': 2024})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 04:02:05 GMT', 'content-type': 'application/json', 'content-length': '169', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '326', 'x-envoy-upstream-service-time': '109', 'x-pinecone-response-duration-ms': '328', 'grpc-status': '0', 'server': 'envoy'}))


# Multiple metadata filters

As well as equality operations,
Pinecone provides operators for other core comparison operations,
as well as enabling multiple filters in a single query.
In this exercise, you'll create multiple filters
using Pinecone's other comparison operators and use them to re-query your index.

In [2]:
from DATA import vector
from pinecone_helper import get_index

index = get_index()

# Retrieve the MOST similar vector with genre and yesr filters
query_result = index.query(
    vector=vector,
    top_k=1,
    filter={
        "genre": "thriller",
        "year": {
            "$lt": 2018,
        },
    },
    include_metadata=True,
)

print(query_result)

QueryResponse(matches=[ScoredVector(id='14', score=0.0561948977, values=[], metadata={'genre': 'thriller', 'year': 2010})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 04:05:38 GMT', 'content-type': 'application/json', 'content-length': '171', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '260', 'x-envoy-upstream-service-time': '45', 'x-pinecone-response-duration-ms': '261', 'grpc-status': '0', 'server': 'envoy'}))


# Updating vector values

In dynamic environments, data evolves rapidly,
and being able to seamlessly integrate new values
and metadata into existing vectors keeps your applications relevant and accurate.

In this exercise, you'll practice updating vectors in the 'datacamp-index' Pinecone index
with new values.
You'll verify these changes were successful by fetching the vector and checking its metadata.

In [3]:
from DATA import vector
from pinecone_helper import get_index

index = get_index()

# Update the values of vector ID 7
index.update(
    id="7",
    values=vector,
)

# Fetch vector ID 7
fetched_vector = index.fetch(ids=["7"])
print(fetched_vector)

FetchResponse(vectors={'7': Vector(id='7', values=[-0.250919759, 0.90142864, 0.463987887, ...1533 more], sparse_values=None, metadata={'genre': 'comedy', 'year': 2007})}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 04:07:58 GMT', 'content-type': 'application/json', 'content-length': '19134', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '47', 'x-envoy-upstream-service-time': '48', 'x-pinecone-response-duration-ms': '50', 'grpc-status': '0', 'server': 'envoy'}))


# Updating vector metadata

This time, you'll practice updating vectors in the datacamp-index Pinecone index
with new metadata.
You'll again verify these changes were successful by retrieving the vector
and checking its metadata.

In [4]:
from pinecone_helper import get_index

index = get_index()

# Update the metadata of vector ID 7
index.update(id="7", set_metadata={"genre": "thriller", "year": 2024})

# Fetch vector ID 7
fetched_vector = index.fetch(ids=["7"])
print(fetched_vector)

FetchResponse(vectors={'7': Vector(id='7', values=[-0.250919759, 0.90142864, 0.463987887, ...1533 more], sparse_values=None, metadata={'genre': 'comedy', 'year': 2007})}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 04:10:08 GMT', 'content-type': 'application/json', 'content-length': '19134', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '42', 'x-envoy-upstream-service-time': '43', 'x-pinecone-response-duration-ms': '44', 'grpc-status': '0', 'server': 'envoy'}))


# Deleting vectors

Deleting vectors isn't just about tidying up our databases;
it's about optimizing performance. As your indexes grow,
unnecessary or outdated vectors can clutter your storage
and slow down query performance.
By removing redundant data, you streamline your operations,
leading to faster responses and better resource utilization.

In this exercise, you'll practice deleting vectors from the 'datacamp-index' Pinecone index.
You'll check the index metrics to verify the deletion took place.

In [5]:
from pinecone_helper import get_index

index = get_index()

# Delete vectors
index.delete(ids=["3", "4"])

# Retrieve metrics of the connected Pinecone index
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=98, metric='cosine', namespaces=1)


# Batching upserts in chunks

In this exercise, you'll practice ingesting vectors
into the 'datacamp-index' Pinecone index in series, batch-by-batch.

In [9]:
import itertools

from DATA import vectors
from pinecone_helper import get_index

Defining a function for chunking.
To be able to batch upserts in a reproducible way,
you'll need to define a function to split your list of vectors into chunks.

In [10]:
def chunks(iterable, batch_size=100):
    """A helper function to break an iterable into chunks of size batch_size."""
    # Convert the iterable into an iterator
    it = iter(iterable)
    # Slice the iterator into chunks of size batch_size
    chunk = tuple(itertools.islice(it, batch_size))
    while chunk:
        # Yield the chunk
        yield chunk
        chunk = tuple(itertools.islice(it, batch_size))

In [11]:
index = get_index()

# Upsert vectors in batches of 100
for chunk in chunks(vectors):
    index.upsert(vectors=chunk)

# Retrieve statistics of the connected Pinecone index
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=100, metric='cosine', namespaces=1)


# Batching upserts in parallel

In this exercise,
you'll practice ingesting vectors into the 'datacamp-index' Pinecone index in parallel.
You'll need to connect to the index, upsert vectors in batches asynchronously,
and check the updated metrics of the 'datacamp-index' index.

Use Pinecone 9.1.10
Pinecone SDK 会自动把 vectors 分成每批 200 个，并使用并发请求上传。
官方文档说明，sync Index.upsert() 会使用 ThreadPoolExecutor 来并行处理 batches。

In [3]:
from DATA import vectors
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()

# Upsert vectors in batches of 200 vectors
with pc.Index("datacamp-index") as index:
    response = index.upsert(
        vectors=vectors,
        batch_size=200,
        max_concurrency=20,
    )
    print(f"Upserted: {response.upserted_count} vectors")

    # Retrieve statistics of the connected Pinecone index
    print(index.describe_index_stats())


Upserting:   0%|          | 0/1 [00:00<?, ?batch/s]

Upserted: 100 vectors
DescribeIndexStatsResponse(dimension=1536, total_vector_count=100, metric='cosine', namespaces=1)


# Namespaces

Namespaces are used to separate vectors within a single index,
allowing targeted queries and minimizing the number of scanned records.

In this exercise, you'll practice ingesting vectors into different namespaces
within the 'datacamp-index' Pinecone index. You'll need to connect to the index,
upsert vectors into different namespaces, and check the updated metrics of the index.

In [4]:
from DATA import vector_set1, vector_set2
from pinecone_helper import get_index

index = get_index()

# Upsert vector_set1 to namespace1
index.upsert(
    vectors=vector_set1,
    namespace="namespace1",
)

# Upsert vector_set2 to namespace2
index.upsert(
    vectors=vector_set2,
    namespace="namespace2",
)

# Print the index statistics
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=200, metric='cosine', namespaces=3)


# Querying namespaces

Now that you've created separate namespaces and embraced multitenancy,
it's time to reap the rewards!
In this exercise, you'll query one of these new namespaces
to search through only a segment of your full index

In [5]:
from DATA import vector
from pinecone_helper import get_index

index = get_index()

# Query namespace1 with the vector provided
query_result = index.query(
    vector=vector,
    namespace="namespace1",
    top_k=3,
)
print(query_result)

QueryResponse(matches=[ScoredVector(id='1001', score=1.00003695, values=[]), ScoredVector(id='1043', score=0.0616056919, values=[]), ScoredVector(id='1015', score=0.0561948977, values=[])], namespace='namespace1', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 04:56:15 GMT', 'content-type': 'application/json', 'content-length': '231', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '281', 'x-envoy-upstream-service-time': '43', 'x-pinecone-response-duration-ms': '282', 'grpc-status': '0', 'server': 'envoy'}))
